# Part 1: Data extraction

## AI PR tables

Downloads the AIDev v3 tables for AI PRs (repositories with more than 100 stars) from Hugging Face into `data/`.

In [1]:
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download

# Download v3 Parquet files (AI PRs)
DATASET_REPO = "hao-li/AIDev"
VERSION = "v3"
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# human_pull_request.parquet is no longer downloaded: the instructor's Human PR
# tables (next cell) contain the same 6,618 PRs with the AIDev-pop layout.
parquet_files = {
    "pull_request_df": "pull_request.parquet",
    "pr_task_type_df": "pr_task_type.parquet",
    "repository_df": "repository.parquet",
    #"user_df": "user.parquet",
    "pr_timeline_df": "pr_timeline.parquet",
    "pr_reviews_df": "pr_reviews.parquet",
    "pr_comments_df": "pr_comments.parquet",
    "pr_review_comments_df": "pr_review_comments_v2.parquet",
    "pr_commits_df": "pr_commits.parquet",
    "pr_commit_details_df": "pr_commit_details.parquet",
    "issue_df": "issue.parquet",
    "related_issue_df": "related_issue.parquet"
}

ai_dataframes = {}

for dataframe_name, filename in parquet_files.items():
    local_path = hf_hub_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        filename=filename,
        revision=VERSION,
        local_dir=DATA_DIR,
    )
    ai_dataframes[dataframe_name] = pd.read_parquet(local_path)
    print(f"Loaded {filename} from {local_path}")

print("AIDev v3 files loaded")

Loaded pull_request.parquet from data/pull_request.parquet
Loaded pr_task_type.parquet from data/pr_task_type.parquet
Loaded repository.parquet from data/repository.parquet
Loaded pr_timeline.parquet from data/pr_timeline.parquet
Loaded pr_reviews.parquet from data/pr_reviews.parquet
Loaded pr_comments.parquet from data/pr_comments.parquet
Loaded pr_review_comments_v2.parquet from data/pr_review_comments_v2.parquet
Loaded pr_commits.parquet from data/pr_commits.parquet
Loaded pr_commit_details.parquet from data/pr_commit_details.parquet
Loaded issue.parquet from data/issue.parquet
Loaded related_issue.parquet from data/related_issue.parquet
AIDev v3 files loaded


## Human PR tables

Loads the Human PR tables given by the instructor. Copy the `AIDev_Human_PRs` folder into `data/` before running this cell (it is not on Hugging Face and is too large for GitHub).

In [2]:
HUMAN_DATA_DIR = DATA_DIR / "AIDev_Human_PRs"

human_parquet_files = {
    "pull_request_df": "pull_request.parquet",
    "pr_task_type_df": "pr_task_type.parquet",
    "repository_df": "repository.parquet",
    "pr_timeline_df": "pr_timeline.parquet",
    "pr_reviews_df": "pr_reviews.parquet",
    "pr_comments_df": "pr_comments.parquet",
    "pr_review_comments_df": "pr_review_comments.parquet",
    "pr_commits_df": "pr_commits.parquet",
    "pr_commit_details_df": "pr_commit_details.parquet",
    "issue_df": "issue.parquet",
    "related_issue_df": "related_issue.parquet"
}

human_dataframes = {}

for dataframe_name, filename in human_parquet_files.items():
    local_path = HUMAN_DATA_DIR / filename
    human_dataframes[dataframe_name] = pd.read_parquet(local_path)
    print(f"Loaded {filename} from {local_path}")

print("Human PR files loaded")

Loaded pull_request.parquet from data/AIDev_Human_PRs/pull_request.parquet
Loaded pr_task_type.parquet from data/AIDev_Human_PRs/pr_task_type.parquet
Loaded repository.parquet from data/AIDev_Human_PRs/repository.parquet
Loaded pr_timeline.parquet from data/AIDev_Human_PRs/pr_timeline.parquet
Loaded pr_reviews.parquet from data/AIDev_Human_PRs/pr_reviews.parquet
Loaded pr_comments.parquet from data/AIDev_Human_PRs/pr_comments.parquet
Loaded pr_review_comments.parquet from data/AIDev_Human_PRs/pr_review_comments.parquet
Loaded pr_commits.parquet from data/AIDev_Human_PRs/pr_commits.parquet
Loaded pr_commit_details.parquet from data/AIDev_Human_PRs/pr_commit_details.parquet
Loaded issue.parquet from data/AIDev_Human_PRs/issue.parquet
Loaded related_issue.parquet from data/AIDev_Human_PRs/related_issue.parquet
Human PR files loaded


## Combine AI and Human tables

Each Human table is aligned to its AI version (same columns, order and types), then added to it. After this cell, `dataframes` has one table per entity with both groups, and `pull_request_df.agent == "Human"` marks Human PRs.

Differences fixed on the way:

| Table | Difference | Fix |
| --- | --- | --- |
| `related_issue_df` | AI `issue_id` stored as a decimal number | Converted to an integer, like `issue_df.id` |
| `pr_task_type_df` | Human PRs have no `confidence` | Left empty for Human PRs |
| `pr_review_comments_df` | `position` has a different type | Human converted to the AI type |
| `repository_df` | Human has an extra `is_forked` column | Dropped |
| `pr_reviews_df` | Human table has one row per review **comment**, with no `state` or `submitted_at` | Grouped to one row per review, like the AI table. `state`, `submitted_at` and `body` stay empty |

812 repositories and 30 issues appear in both groups. We keep the AI row for those (star counts differ slightly because the data was collected at different times).

In [3]:
def align_to(reference_df, df):
    """Return df with the columns, column order and dtypes of reference_df."""
    extra_columns = set(df.columns) - set(reference_df.columns)
    assert not extra_columns, f"Unexpected columns: {extra_columns}"
    return df.reindex(columns=reference_df.columns).astype(reference_df.dtypes.to_dict())


ai_dataframes["related_issue_df"]["issue_id"] = ai_dataframes["related_issue_df"]["issue_id"].astype("int64")
ai_dataframes["pr_task_type_df"]["confidence"] = ai_dataframes["pr_task_type_df"]["confidence"].astype("Int64")

human_dataframes["pr_review_comments_df"]["position"] = human_dataframes["pr_review_comments_df"]["position"].astype("float64")
human_dataframes["repository_df"] = human_dataframes["repository_df"].drop(columns="is_forked")
human_dataframes["pr_reviews_df"] = (
    human_dataframes["pr_reviews_df"]
    .groupby("pull_request_review_id", as_index=False)
    .agg(pr_id=("pr_id", "first"), user=("user", "first"), user_type=("user_type", "first"))
    .rename(columns={"pull_request_review_id": "id"})
)

# Tables keyed by id that describe entities shared by both groups: keep the AI row.
shared_entity_tables = ["repository_df", "issue_df"]

dataframes = {}

for dataframe_name, ai_df in ai_dataframes.items():
    human_df = align_to(ai_df, human_dataframes[dataframe_name])
    combined_df = pd.concat([ai_df, human_df], ignore_index=True)
    if dataframe_name in shared_entity_tables:
        combined_df = combined_df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
    dataframes[dataframe_name] = combined_df
    print(f"{dataframe_name:25} AI: {len(ai_df):7}  Human: {len(human_df):7}  Combined: {len(combined_df):7}")

for dataframe_name, dataframe in dataframes.items():
    assert dataframe.dtypes.equals(ai_dataframes[dataframe_name].dtypes), dataframe_name

print("AI and Human tables combined")

pull_request_df           AI:   33596  Human:    6618  Combined:   40214
pr_task_type_df           AI:   33596  Human:    6618  Combined:   40214
repository_df             AI:    2807  Human:     818  Combined:    2813
pr_timeline_df            AI:  325500  Human:   91012  Combined:  416512
pr_reviews_df             AI:   28875  Human:    6328  Combined:   35203
pr_comments_df            AI:   39122  Human:   11236  Combined:   50358
pr_review_comments_df     AI:   26868  Human:    9311  Combined:   36179
pr_commits_df             AI:   88576  Human:   26991  Combined:  115567
pr_commit_details_df      AI:  711923  Human:  300693  Combined: 1012616
issue_df                  AI:    4614  Human:     847  Combined:    5431
related_issue_df          AI:    4923  Human:    1071  Combined:    5994
AI and Human tables combined


Quick look at the combined tables: number of rows and columns, and the first rows of each table.

In [4]:

def table_inventory(dataframes):
    rows = []
    for table_name, dataframe in dataframes.items():
        rows.append(
            {
                "table": table_name,
                "rows": len(dataframe),
                "columns": len(dataframe.columns),
                "column_names": ", ".join(dataframe.columns),
            }
        )
    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)

inventory = table_inventory(dataframes)
# pd.set_option("display.max_colwidth", 160)
display(inventory)

for table_name, dataframe in dataframes.items():
    print(f"\n=== {table_name} ===")
    display(dataframe.head())

,table,rows,columns,column_names
0,issue_df,5431,9,"id, number, title, body, user, state, created_..."
1,pr_comments_df,50358,7,"id, pr_id, user, user_id, user_type, created_a..."
2,pr_commit_details_df,1012616,14,"sha, pr_id, author, committer, message, commit..."
3,pr_commits_df,115567,5,"sha, pr_id, author, committer, message"
4,pr_review_comments_df,36179,15,"id, pull_request_review_id, user, user_type, d..."
5,pr_reviews_df,35203,7,"id, pr_id, user, user_type, state, submitted_a..."
6,pr_task_type_df,40214,6,"agent, id, title, reason, type, confidence"
7,pr_timeline_df,416512,8,"pr_id, event, commit_id, created_at, actor, as..."
8,pull_request_df,40214,14,"id, number, title, body, agent, user_id, user,..."
9,related_issue_df,5994,3,"pr_id, issue_id, source"



=== pull_request_df ===


,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url
0,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,108661493,weiliu1031,closed,2025-07-26T02:59:01Z,2025-07-29T07:01:20Z,None,191751505,https://api.github.com/repos/milvus-io/pymilvus,https://github.com/milvus-io/pymilvus/pull/2911
1,3265118634,2,ファイルパス参照を相対パスに統一し、doc/からdocs/に統一,## 背景\n\n現在、本プロジェクトにおいて以下のパス構成の不整合が生じています：\n\n...,Claude_Code,61827001,cm-kojimat,closed,2025-07-26T04:56:55Z,2025-07-26T22:12:24Z,2025-07-26T22:12:24Z,1025472321,https://api.github.com/repos/classmethod/tsumiki,https://github.com/classmethod/tsumiki/pull/2
2,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,7475,MSch,closed,2025-07-26T13:31:19Z,2025-07-26T13:37:22Z,2025-07-26T13:37:22Z,988488798,https://api.github.com/repos/steipete/Peekaboo,https://github.com/steipete/Peekaboo/pull/30
3,3265709660,205,feat: add comprehensive README screenshots wit...,## Type of Change\n\n- [ ] 🐛 `bug` - Bug fix (...,Claude_Code,80381,sugyan,closed,2025-07-26T14:07:22Z,2025-07-26T14:45:30Z,2025-07-26T14:45:30Z,999285986,https://api.github.com/repos/sugyan/claude-cod...,https://github.com/sugyan/claude-code-webui/pu...
4,3265782173,17625,chore: remove HashedPostStateProvider trait,## Summary\r\n\r\n#17545 \r\n\r\nRemove the un...,Claude_Code,47593288,adust09,open,2025-07-26T15:02:48Z,None,None,537233603,https://api.github.com/repos/paradigmxyz/reth,https://github.com/paradigmxyz/reth/pull/17625



=== pr_task_type_df ===


,agent,id,title,reason,type,confidence
0,Claude_Code,3264933329,Fix: Wait for all partitions in load_collectio...,title provides conventional commit label,fix,10
1,Claude_Code,3265709660,feat: add comprehensive README screenshots wit...,title provides conventional commit label,feat,10
2,Claude_Code,3265782173,chore: remove HashedPostStateProvider trait,title provides conventional commit label,chore,10
3,Claude_Code,3231949586,feat(swagger): Add Swagger annotations to Batc...,title provides conventional commit label,feat,10
4,Claude_Code,3231950376,feat(swagger): Add Swagger annotations to Batc...,title provides conventional commit label,feat,10


...

,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,additions,deletions,changes,patch
0,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,.github/workflows/compile-llms-txt.yml,added,38.0,0.0,38.0,"@@ -0,0 +1,38 @@\n+name: Compile llms.txt\n+\n..."
1,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,docs/compile_llms_txt.py,added,47.0,0.0,47.0,"@@ -0,0 +1,47 @@\n+import os\n+from pathlib im..."
2,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,llms.txt,added,22923.0,0.0,22923.0,None
3,dbd1b5f129f7cffa5ce284d7255814c98bcc38a2,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Fix lint issues: remove unused variable and ap...,35,18,17,docs/compile_llms_txt.py,modified,18.0,17.0,35.0,"@@ -1,47 +1,48 @@\n import os\n from pathlib i..."
4,c2659cfdedf666c8f14753d71664563c2a932b23,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Update llms.txt to follow official standard wi...,23035,89,22946,docs/compile_llms_txt.py,modified,51.0,36.0,87.0,"@@ -3,45 +3,60 @@\n \n \n def compile_llms_txt..."



=== issue_df ===


,id,number,title,body,user,state,created_at,closed_at,html_url
0,3082498013,1542,Accept api_key in from_provider,**Is your feature request related to a problem...,jeroenvds,closed,2025-05-22T08:22:22Z,2025-07-17T22:34:21Z,https://github.com/567-labs/instructor/issues/...
1,3164229012,94,Workflows Coming Soon - tools reimagined,### Project Version\n\n5.5.0\n\n### Bug Descri...,guidedways,closed,2025-06-20T19:42:16Z,2025-06-20T20:08:12Z,https://github.com/BeehiveInnovations/zen-mcp-...
2,3198054361,107,[bug]PromptX工具沙箱缓存机制不支持工具集更新,"当工具集需要改进的时候, 会遇到改进后代码和沙箱中的代码不一致的情况.对鲁班自己调试代码造成...",simonfishgit,closed,2025-07-03T05:20:38Z,2025-07-10T01:44:56Z,https://github.com/Deepractice/PromptX/issues/107
3,3208181218,129,架构优化：统一项目路径管理体系 - 合并DirectoryService与ProjectMa...,# 🎯 架构优化：统一项目路径管理体系\n\n## 🔍 问题描述\n\n当前PromptX在...,deepracticexs,closed,2025-07-07T09:12:24Z,2025-07-07T11:05:45Z,https://github.com/Deepractice/PromptX/issues/129
4,2975033421,836,Broken links to notebooks in documentation,On [this documentation page](https://incatools...,justaddcoffee,closed,2025-04-06T15:46:33Z,2025-06-05T23:32:27Z,https://github.com/INCATools/ontology-access-k...



=== related_issue_df ===


,pr_id,issue_id,source
0,3234102722,3233112858,body
1,3214782537,3189852905,body
2,3164567685,3164538512,body
3,3165176791,2510820749,body
4,3165431934,3165438347,body


# Part 2: Data cleaning and classification

### Q1: How you removed duplicates

1. **Same PR in both groups.** 56 Human PRs are also in the AI data (same `html_url`, different `id`). They are agent PRs, so we kept the AI version and removed the Human copy, with all its linked rows (commits, files, reviews, comments, timeline, linked issues).
2. **Rows that appear twice.** We removed exact copies:

| Table | Rows removed | Note |
| --- | --- | --- |
| `pr_commits_df` | 12 | |
| `pr_commit_details_df` | 23 | Rows where a file was removed and added back in the same commit are kept |
| `related_issue_df` | 50 | Some Human PR–issue links were found twice (in the PR description and in the timeline). We keep one, from the description, the only source used for AI PRs |
| `pr_timeline_df` | 12 | Only repeated `committed` events. Other identical rows can be real separate events (the timeline has no event id), so they are kept |

The code then checks that no id appears twice in the tables that have one.

### Q2: How you handled missing values

Most missing values have a meaning, so we kept them instead of guessing a value:

- **Dates** are converted to UTC. An empty `closed_at` means the PR or issue is still open. An empty `merged_at` means the PR was not merged. The code checks that this holds for every PR and issue.
- **Text** (PR or issue description, review text, commit message): empty values become `""`, since nothing was written.
- **Left empty on purpose:** fields missing from the Human data (`confidence`, review `state` and `submitted_at`), commits that changed no files, very large diffs that GitHub does not return (`patch`), timeline columns that only apply to some events, and repository `license` / `language` (not used).
- **29 PRs have no commit data.** They are kept for merge rate and time to merge, but will be left out of the size metrics.

After cleaning: 33,596 AI PRs and 6,562 Human PRs.

In [ ]:
cleaning_log = []


def log_step(step, table_name, rows_before, rows_after):
    cleaning_log.append({"step": step, "table": table_name, "rows_removed": rows_before - rows_after, "rows_after": rows_after})


# Treat Human PRs that are AIDev agent PRs (same html_url, different id): keep the AI version and drop
# the Human copy with all its child rows.
pull_request_df = dataframes["pull_request_df"]
is_human = pull_request_df["agent"] == "Human"
ai_urls = set(pull_request_df.loc[~is_human, "html_url"])
duplicate_pr_ids = set(pull_request_df.loc[is_human & pull_request_df["html_url"].isin(ai_urls), "id"])
duplicate_review_ids = set(dataframes["pr_reviews_df"].loc[dataframes["pr_reviews_df"]["pr_id"].isin(duplicate_pr_ids), "id"])

for dataframe_name, dataframe in dataframes.items():
    if dataframe_name in shared_entity_tables:
        continue
    if dataframe_name in ["pull_request_df", "pr_task_type_df"]:
        is_duplicate_pr = dataframe["id"].isin(duplicate_pr_ids)
    elif dataframe_name == "pr_review_comments_df":
        is_duplicate_pr = dataframe["pull_request_review_id"].isin(duplicate_review_ids)
    else:
        is_duplicate_pr = dataframe["pr_id"].isin(duplicate_pr_ids)
    dataframes[dataframe_name] = dataframe[~is_duplicate_pr].reset_index(drop=True)
    log_step("Human copy of an agent PR", dataframe_name, len(dataframe), len(dataframes[dataframe_name]))

# Exact duplicate rows treatment
for dataframe_name, dataframe in dataframes.items():
    if dataframe_name == "pr_timeline_df":
        is_duplicate = (dataframe["event"] == "committed") & dataframe.duplicated(subset=["pr_id", "event", "commit_id"])
    elif dataframe_name == "related_issue_df":
        dataframe = dataframe.sort_values("source", kind="stable")
        is_duplicate = dataframe.duplicated(subset=["pr_id", "issue_id"])
    else:
        is_duplicate = dataframe.duplicated()
    dataframes[dataframe_name] = dataframe[~is_duplicate].reset_index(drop=True)
    log_step("Exact duplicate row", dataframe_name, len(dataframe), len(dataframes[dataframe_name]))

primary_keys = {
    "pull_request_df": ["id"],
    "pr_task_type_df": ["id"],
    "repository_df": ["id"],
    "issue_df": ["id"],
    "pr_reviews_df": ["id"],
    "pr_comments_df": ["id"],
    "pr_review_comments_df": ["id"],
    "pr_commits_df": ["sha", "pr_id"],
    "related_issue_df": ["pr_id", "issue_id"],
}
for dataframe_name, key_columns in primary_keys.items():
    assert not dataframes[dataframe_name].duplicated(subset=key_columns).any(), f"Duplicate key in {dataframe_name}"

# Treat missing timestamps
timestamp_columns = {
    "pull_request_df": ["created_at", "closed_at", "merged_at"],
    "issue_df": ["created_at", "closed_at"],
    "pr_timeline_df": ["created_at"],
    "pr_reviews_df": ["submitted_at"],
    "pr_comments_df": ["created_at"],
    "pr_review_comments_df": ["created_at", "updated_at"],
}
for dataframe_name, columns in timestamp_columns.items():
    for column in columns:
        parsed = pd.to_datetime(dataframes[dataframe_name][column], utc=True, errors="coerce")
        assert parsed.notna().equals(dataframes[dataframe_name][column].notna()), f"Unparsable dates in {dataframe_name}.{column}"
        dataframes[dataframe_name][column] = parsed

# Treat missing text (replace with empty strings)
text_columns = {
    "pull_request_df": ["body"],
    "issue_df": ["body"],
    "pr_reviews_df": ["body"],
    "pr_review_comments_df": ["body"],
    "pr_commits_df": ["message"],
    "pr_commit_details_df": ["message"],
}
for dataframe_name, columns in text_columns.items():
    dataframes[dataframe_name][columns] = dataframes[dataframe_name][columns].fillna("")

# Missing timestamps must match the PR and issue states
pull_request_df = dataframes["pull_request_df"]
assert pull_request_df["closed_at"].isna().equals(pull_request_df["state"] == "open")
assert pull_request_df.loc[pull_request_df["merged_at"].notna(), "closed_at"].notna().all()
assert (pull_request_df["merged_at"].dropna() >= pull_request_df.loc[pull_request_df["merged_at"].notna(), "created_at"]).all()
assert (pull_request_df["closed_at"].dropna() >= pull_request_df.loc[pull_request_df["closed_at"].notna(), "created_at"]).all()
assert dataframes["issue_df"]["closed_at"].isna().equals(dataframes["issue_df"]["state"] == "open")

cleaning_summary = pd.DataFrame(cleaning_log)
display(cleaning_summary[cleaning_summary["rows_removed"] > 0].reset_index(drop=True))

remaining_missing = pd.DataFrame(
    [
        {"table": dataframe_name, "column": column, "missing": count, "pct": round(100 * count / len(dataframe), 2)}
        for dataframe_name, dataframe in dataframes.items()
        for column, count in dataframe.isna().sum().items()
        if count > 0
    ]
)
display(remaining_missing)

prs_without_commits = ~pull_request_df["id"].isin(dataframes["pr_commits_df"]["pr_id"])
print(f"PRs without commit data (kept, excluded later from size metrics): {prs_without_commits.sum()}")
print(pull_request_df["agent"].eq("Human").map({True: "Human", False: "AI"}).value_counts().to_string())

,step,table,rows_removed,rows_after
0,Human copy of an agent PR,pull_request_df,56,40158
1,Human copy of an agent PR,pr_task_type_df,56,40158
2,Human copy of an agent PR,pr_timeline_df,643,415869
3,Human copy of an agent PR,pr_reviews_df,32,35171
4,Human copy of an agent PR,pr_comments_df,119,50239
5,Human copy of an agent PR,pr_review_comments_df,72,36107
6,Human copy of an agent PR,pr_commits_df,164,115403
7,Human copy of an agent PR,pr_commit_details_df,997,1011619
8,Human copy of an agent PR,related_issue_df,10,5984
9,Exact duplicate row,pr_timeline_df,12,415857


,table,column,missing,pct
0,pull_request_df,closed_at,2762,6.88
1,pull_request_df,merged_at,11094,27.63
2,pr_task_type_df,confidence,6562,16.34
3,repository_df,license,191,6.79
4,repository_df,language,37,1.32
5,pr_timeline_df,commit_id,253169,60.88
6,pr_timeline_df,created_at,127551,30.67
7,pr_timeline_df,actor,23379,5.62
8,pr_timeline_df,assignee,402775,96.85
9,pr_timeline_df,label,381284,91.69


PRs without commit data (kept, excluded later from size metrics): 29
agent
AI       33596
Human     6562


### Q3: How you identified AI-related vs. human-related contributions

**AI PRs:** we used the AIDev labels (`agent` column). We checked that each agent's trace is really in the data: Claude Code footer in the description (100% of its PRs), PR opened by `devin-ai-integration[bot]` (100%) or `Copilot` (100%), Codex link in the description (80%), commits by `cursoragent` (97%).

**Human PRs:** the Human data is not all human, so the code looks at who opened each PR and for AI traces in the description and commits. The result is the `ai_involvement` column:

| `ai_involvement` | Meaning | PRs |
| --- | --- | --- |
| `human_only` | Human PR, no AI trace | 5,436 |
| `human_with_ai` | Human PR with some commits written by an agent | 129 |
| `agent_unlabeled` | Agent PR that AIDev missed (agent trace in the description) | 81 |
| `bot` | Opened by a bot (dependabot, renovate, ...) | 885 |
| `other_ai` | Opened by an AI agent that is not one of the 5 AIDev agents | 31 |
| `agent` | AIDev agent PR | 33,596 |

The code also adds:
- `has_human_commits`: AI PR where a person also pushed commits (29% of Copilot, 31% of Devin and 41% of Cursor PRs). Not measurable for Codex and Claude Code, which commit under the developer's account.
- `actor_type` on reviews and comments: `human`, `ai_bot` or `other_bot`. 40% of reviews on AI PRs come from AI bots.
- `in_matched_subset`: AI PRs from the same repositories and period as the Human PRs (7,541), for a fair comparison.

See *Notes for Parts 3 and 4* at the end of the notebook for how to use these columns.

In [7]:
pull_request_df = dataframes["pull_request_df"]
pr_commits_df = dataframes["pr_commits_df"]

pull_request_df["group"] = pull_request_df["agent"].where(pull_request_df["agent"] == "Human", "AI")
is_human_pr = pull_request_df["group"] == "Human"

# Bot accounts without the "[bot]" suffix (logins containing "bot" checked by hand)
OTHER_BOT_LOGINS = {
    "CBL-Mariner-Bot", "SyliusBot", "calcom-bot", "ckanbot", "docs-bot", "dotnet-bot", "kiwicom-orbit-bot",
    "modularbot", "vercel-release-bot", "xerial-bot", "youssef-backport-bot",
}
# AI coding agents that are not one of the five AIDev agents
OTHER_AI_LOGINS = {
    "gru-agent[bot]", "codeflash-ai[bot]", "tembo-io[bot]", "dane-ai-mastra[bot]",
    "entelligence-ai-pr-reviews[bot]", "RepoBirdBot",
}

author_type = pd.Series("agent", index=pull_request_df.index)
author_type[is_human_pr] = "human"
author_type[is_human_pr & (pull_request_df["user"].str.endswith("[bot]") | pull_request_df["user"].isin(OTHER_BOT_LOGINS))] = "bot"
author_type[is_human_pr & pull_request_df["user"].isin(OTHER_AI_LOGINS)] = "other_ai"
pull_request_df["author_type"] = author_type


# Signatures the agents leave in PR descriptions and commit messages (PRs co-authered Human-AI)
AI_SIGNATURE_PATTERN = (
    r"Generated with \[?Claude Code|claude\.ai/code|Co-Authored-By: Claude|copilot-swe-agent|"
    r"Co-authored-by: Copilot|chatgpt\.com/codex|cursoragent|devin\.ai|app\.devin"
)
# Accounts that agents use to author or commit code
AGENT_COMMIT_ACCOUNTS = {
    "Copilot", "copilot-swe-agent[bot]", "devin-ai-integration[bot]", "cursoragent", "claude[bot]",
    "google-labs-jules[bot]", "gru-agent[bot]", "codeflash-ai[bot]",
}

has_description_trace = (
    pull_request_df["title"].str.contains(AI_SIGNATURE_PATTERN, case=False, regex=True)
    | pull_request_df["body"].str.contains(AI_SIGNATURE_PATTERN, case=False, regex=True)
)
is_ai_commit = (
    pr_commits_df["message"].str.contains(AI_SIGNATURE_PATTERN, case=False, regex=True)
    | pr_commits_df["author"].isin(AGENT_COMMIT_ACCOUNTS)
    | pr_commits_df["committer"].isin(AGENT_COMMIT_ACCOUNTS)
)
has_commit_trace = pull_request_df["id"].isin(pr_commits_df.loc[is_ai_commit, "pr_id"])

# AI involvement level of every PR

ai_involvement = author_type.copy()                   
ai_involvement[author_type == "human"] = "human_only"
ai_involvement[(author_type == "human") & has_commit_trace] = "human_with_ai"
ai_involvement[(author_type == "human") & has_description_trace] = "agent_unlabeled"
pull_request_df["ai_involvement"] = ai_involvement

# Human commits in AI PRs
AGENT_OWN_ACCOUNTS = {
    "Copilot": {"Copilot", "copilot-swe-agent[bot]"},
    "Devin": {"devin-ai-integration[bot]"},
    "Cursor": {"cursoragent"},
}
commit_agent = pr_commits_df["pr_id"].map(pull_request_df.set_index("id")["agent"])
is_human_commit = pd.Series(False, index=pr_commits_df.index)
for agent, accounts in AGENT_OWN_ACCOUNTS.items():
    is_human_commit |= (
        (commit_agent == agent)
        & pr_commits_df["author"].notna()
        & ~pr_commits_df["author"].isin(accounts)
        & ~pr_commits_df["author"].str.endswith("[bot]", na=False)
    )

has_commits = pull_request_df["id"].isin(pr_commits_df["pr_id"])
has_human_commits = pd.Series(pd.NA, index=pull_request_df.index, dtype="boolean")
measurable = pull_request_df["agent"].isin(AGENT_OWN_ACCOUNTS.keys()) & has_commits
has_human_commits[measurable] = pull_request_df.loc[measurable, "id"].isin(pr_commits_df.loc[is_human_commit, "pr_id"])
pull_request_df["has_human_commits"] = has_human_commits

# Matched subset: AI PRs from the same repositories and period as the Human PRs

human_repo_ids = set(pull_request_df.loc[is_human_pr, "repo_id"])
human_period_start = pull_request_df.loc[is_human_pr, "created_at"].min()
human_period_end = pull_request_df.loc[is_human_pr, "created_at"].max()
pull_request_df["in_matched_subset"] = (
    pull_request_df["repo_id"].isin(human_repo_ids)
    & pull_request_df["created_at"].between(human_period_start, human_period_end)
)


# AI agents and AI code review bots
AI_BOT_LOGINS = {
    "Copilot", "copilot-swe-agent[bot]", "copilot-pull-request-reviewer[bot]", "devin-ai-integration[bot]",
    "cursor[bot]", "claude[bot]", "coderabbitai[bot]", "greptile-apps[bot]", "gemini-code-assist[bot]",
    "qodo-merge-for-open-source[bot]", "qodo-merge-pro[bot]", "ellipsis-dev[bot]", "entelligence-ai-pr-reviews[bot]",
    "cubic-dev-ai[bot]", "sourcery-ai[bot]", "gru-agent[bot]", "recurseml[bot]", "seer-by-sentry[bot]",
    "jazzberry-ai[bot]", "kodus-ai[bot]", "use-tusk[bot]", "codeflash-ai[bot]", "google-labs-jules[bot]",
}


def classify_actor(dataframe):
    is_bot = (dataframe["user_type"] == "Bot") | dataframe["user"].str.contains(r"(?:\[bot\]|bot)$", case=False, regex=True, na=False)
    actor_type = pd.Series("human", index=dataframe.index)
    actor_type[is_bot] = "other_bot"
    actor_type[dataframe["user"].isin(AI_BOT_LOGINS)] = "ai_bot"
    actor_type[dataframe["user"].isna()] = pd.NA
    return actor_type


for dataframe_name in ["pr_reviews_df", "pr_comments_df", "pr_review_comments_df"]:
    dataframes[dataframe_name]["actor_type"] = classify_actor(dataframes[dataframe_name])

display(pd.crosstab(pull_request_df["ai_involvement"], pull_request_df["group"], margins=True))
display(
    pull_request_df[pull_request_df["group"] == "AI"]
    .groupby("agent")
    .agg(
        prs=("id", "size"),
        in_matched_subset=("in_matched_subset", "sum"),
        with_human_commits=("has_human_commits", "sum"),
        human_commits_measured=("has_human_commits", "count"),
    )
)
pr_group = pull_request_df.set_index("id")["group"]
display(
    pd.concat(
        {
            dataframe_name: pd.crosstab(
                dataframes[dataframe_name]["pr_id"].map(pr_group), dataframes[dataframe_name]["actor_type"], normalize="index"
            ).round(3)
            for dataframe_name in ["pr_reviews_df", "pr_comments_df"]
        }
    )
)

group,AI,Human,All
ai_involvement,,,
agent,33596,0,33596
agent_unlabeled,0,81,81
bot,0,885,885
human_only,0,5436,5436
human_with_ai,0,129,129
other_ai,0,31,31
All,33596,6562,40158


,prs,in_matched_subset,with_human_commits,human_commits_measured
agent,,,,
Claude_Code,459,110,0,0
Copilot,4970,1707,1463,4967
Cursor,1541,196,625,1540
Devin,4827,2757,1476,4822
OpenAI_Codex,21799,2771,0,0


actor_type            ai_bot  human  other_bot
               pr_id                          
pr_reviews_df  AI      0.398  0.585      0.017
               Human   0.166  0.819      0.015
pr_comments_df AI      0.416  0.296      0.288
               Human   0.126  0.435      0.439

### Q4: How you classified task types

1. **Task type of each PR.** The AIDev dataset already gives every PR, AI and Human, one type (`feat`, `fix`, `docs`, `test`, ...), chosen by an LLM from the PR title and description. We copy it into `pull_request_df` as `task_type`.
2. **Categories from the lab instructions.** We group the types into the `task_category` column:

| AIDev type | Category |
| --- | --- |
| `fix` | Bug fixes |
| `feat` | Feature implementation |
| `docs` | Documentation |
| `refactor`, `style`, `perf` | Refactoring |
| `test` | Testing |
| `ci`, `build` | CI/CD or configuration |
| `chore`, `revert`, `other` | Maintenance |

No AIDev type matches "Experimental or research-related changes", so that category is not used.

In [ ]:
pull_request_df = dataframes["pull_request_df"]
pr_task_type_df = dataframes["pr_task_type_df"]

# Task type from AIDev (LLM classification into Conventional Commit types)
pull_request_df["task_type"] = pull_request_df["id"].map(pr_task_type_df.set_index("id")["type"])
assert pull_request_df["task_type"].notna().all()

# Assignment categories (wording from the lab PDF). No AIDev type matches
# "Experimental or research-related changes", so that category stays empty.
TASK_CATEGORIES = {
    "fix": "Bug fixes",
    "feat": "Feature implementation",
    "docs": "Documentation",
    "refactor": "Refactoring",
    "style": "Refactoring",
    "perf": "Refactoring",
    "test": "Testing",
    "ci": "CI/CD or configuration",
    "build": "CI/CD or configuration",
    "chore": "Maintenance",
    "revert": "Maintenance",
    "other": "Maintenance",
}
pull_request_df["task_category"] = pull_request_df["task_type"].map(TASK_CATEGORIES)
assert pull_request_df["task_category"].notna().all()

display(pd.crosstab(pull_request_df["task_category"], pull_request_df["group"], margins=True))

### Q5: Any assumptions you made

**Data**
- The instructor's Human tables describe the same 6,618 PRs as AIDev's `human_pull_request` table (we checked: same ids and values, except that some descriptions use different line-break characters).
- Human reviews only include reviews that left inline comments, and have no review state.

**Duplicates and missing values**
- Two PRs with the same `html_url` are the same PR. When it is in both groups, the AI label is kept.
- Identical timeline rows can be separate real events, so only repeated `committed` events are removed.
- An empty `closed_at` means "still open", an empty `merged_at` means "not merged", and empty text means "nothing written".

**AI vs. human**
- AIDev's agent labels are correct.
- A Human PR with no visible AI trace was written without AI. Someone can use AI without leaving a trace, so `human_only` really means "no visible AI".
- The bot and AI account lists and the trace patterns in the code were built from what we found in this data.

**Task types**
- Each PR has one main type. A PR that adds a feature and its tests counts as a feature.
- The LLM labels are correct. We did not check them by hand; this goes in the threats to validity.

## Notes for Parts 3 and 4

### New columns

No rows were removed when labelling. Each analysis picks the rows it needs.

**`pull_request_df`**

| Column | Values | Meaning |
| --- | --- | --- |
| `group` | `AI`, `Human` | Which dataset the PR comes from |
| `author_type` | `agent`, `human`, `bot`, `other_ai` | Who opened the PR |
| `ai_involvement` | see Part 2 | How much AI was involved |
| `has_human_commits` | `True`, `False`, empty | AI PR with at least one commit by a person. Empty for Codex, Claude Code and PRs with no commit data |
| `in_matched_subset` | `True`, `False` | PR from the same repositories and period as the Human PRs |
| `task_type` | `feat`, `fix`, ... | AIDev task type |
| `task_category` | `Bug fixes`, ... | Task category from the lab instructions |

**`pr_reviews_df`, `pr_comments_df`, `pr_review_comments_df`**: `actor_type` = `human`, `ai_bot` or `other_bot` (empty when the account was deleted).

### How to use them

- **Leave out** `bot` and `other_ai` PRs from the RQ1 and RQ2 comparisons.
- **Levels of AI involvement**, from none to full: `human_only` → `human_with_ai` → `agent` (split with `has_human_commits` for Copilot, Devin and Cursor). `agent_unlabeled` can be counted as `agent`.
- **Review rounds (RQ1):** count only reviews with `actor_type == "human"`. Otherwise the agent reviewing its own PR counts as review activity (the Copilot agent alone posts 6,197 reviews on its own PRs).
- **Task types:** compare merge rate and time to merge within each `task_category` too. AI PRs have more features (43% vs 33%), so a gap between AI and Human could come from the kind of work rather than from who did it.

### Why the matched subset matters

Human PRs come only from 818 repositories with more than 500 stars, created between January and June 2025. AI PRs come from 2,807 repositories with more than 100 stars, and 92% were created in May–July 2025.

Rates (merge rate, median time to merge, lines per PR) fix the fact that the AI group is bigger. They do **not** fix the fact that the groups come from different projects. The AI merge rate is 77% on all AI PRs but 60% on AI PRs from the same repositories and period as the Human PRs (Human: 85%). One user, `tamnd`, opened 8,913 Codex PRs in one personal repository, and 87% were merged. That is 27% of all AI PRs.

**Suggestion:**
- Main results on `in_matched_subset == True` (7,541 AI PRs and the Human PRs).
- Same analysis on all PRs as a check. If both agree, the result is solid. If not, say so in the discussion.
- Results per agent too (`agent` column), since the five agents behave very differently.

## Metric plan and intended table usage


| Research question | Metric or variable | Source table(s) | Join key |
| --- | --- | --- | --- |
| RQ1: development outcomes | Merge rate, time to merge pull request | `pull_request_df` (`agent` column: agent name or `Human`) | PR `id` identifies the AI/human group; `pr_id` in related tables (want to look at same repos? could do analysis of both complete datasets even if repos not 1 to 1 match in all cases) |
| RQ1: development outcomes | Issue closing rate, time to close issue | `issue_df`, `related_issue_df`, `pull_request_df` (`agent` column: agent name or `Human`) | `related_issue_df.issue_id = issue_df.id`; `related_issue_df.pr_id = pull_request.id` |
| RQ1: development outcomes | Number of review rounds, number of commits before acceptance | `pr_reviews_df`, `pr_commits_df`, `pull_request_df` | `pull_request_df.id = pr_reviews_df.pr_id = pr_commits_df.pr_id` |
| RQ1: development outcomes | Reopened issues, revised pull requests | `issue_df`, `related_issue_df`, `pr_timeline_df`, `pull_request_df` | Issues use `issue_id`; PR history uses `pull_request_df.id = pr_timeline_df.pr_id` |
| RQ2: contribution characteristics | Files changed, lines added, lines deleted, code churn, number of commits | `pr_commit_details_df`, `pr_commits_df`, `pull_request_df` | `pull_request_df.id = pr_commit_details_df.pr_id = pr_commits_df.pr_id` |
| RQ2: contribution characteristics | Types of files modified, documentation versus code, test-related changes, configuration or infrastructure changes | `pr_commit_details_df`, `pull_request_df` | `pull_request_df.id = pr_commit_details_df.pr_id` |


All tables contain both AI and Human rows. Human PRs come from repositories with more than 500 stars, while AIDev AI PRs use repositories with more than 100 stars.